# 02 · Fase 2 — Do protótipo ao agente: ferramentas, ReAct, roteamento e prompts

Rubricas **2.1 a 2.6**: comparação de orquestradores, agente ReAct com ferramentas e memória, chains sequenciais/paralelas e roteamento dinâmico, API externa real via function calling, prompt PTCF com few-shot e chain-of-thought, e componentes de confiabilidade (parser com reparo, memória configurada, FAISS, callbacks).

> **Sobre o modo offline.** O enunciado pede alternativas *dummy* para APIs e credenciais. Sem `OPENAI_API_KEY` (ou com `MODO_OFFLINE=true`), o projeto troca o GPT por um **modelo roteirizado** (`src/llm.py`: regras determinísticas que escolhem a ferramenta e formatam a observação), os embeddings por `DeterministicFakeEmbedding` e as APIs por **fixtures gravados** de chamadas reais (`fixtures/`). A estrutura das chains, do agente, da memória e da verificação é exatamente a mesma; só a "inteligência" muda. As saídas deste notebook foram geradas no modo indicado na célula de configuração.

## 2.1 Orquestradores: por que LangChain

| Framework | Autonomia (quem decide o próximo passo) | Abstração | Modularidade | Observação |
|---|---|---|---|---|
| **LangChain + LangGraph** | LLM decide via tool calling; grafo explícito quando se quer controle | Média: `Runnable` unificado, `create_agent` pronto | Alta: troca LLM, ferramentas, memória e vector store sem reescrever | Tracing nativo no LangSmith; a próxima disciplina continua em LangGraph |
| LlamaIndex | Focado em ingestão/consulta de documentos | Alta para RAG, menor para agentes | Média | Ótimo para RAG puro; agente e memória são secundários |
| AutoGen | Multiagente conversacional | Alta | Média | Bom para times de agentes; excesso para um agente só |
| Semantic Kernel | Plugins/planners (ecossistema .NET/C#) | Média | Média | Integração Microsoft; comunidade Python menor |
| Langflow / Flowise | Visual, low-code | Muito alta | Baixa | Prototipagem; difícil versionar e testar |

**Decisão:** LangChain. A interface `Runnable` cobre chains, ferramentas e o agente com a mesma API; `create_agent` traz o ciclo ReAct com checkpointer; o LangSmith avalia trajetórias sem instrumentação extra; e o agente desta disciplina é o ponto de partida da próxima (LangGraph).

In [1]:
# Configuração comum: roda a partir da raiz do repositório e mostra o modo de execução
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
import warnings

warnings.filterwarnings("ignore")

from src import config, llm

print(
    "LangChain",
    version("langchain"),
    "| langchain-core",
    version("langchain-core"),
    "| LangGraph",
    version("langgraph"),
    "| LangSmith",
    version("langsmith"),
)
print(
    "Modo:",
    "OFFLINE (modelo dummy roteirizado + fixtures)" if llm.offline() else f"ONLINE ({config.MODELO_AGENTE})",
)

LangChain 1.4.9 | langchain-core 1.7.6 | LangGraph 1.2.20 | LangSmith 0.14.17
Modo: ONLINE (gpt-4o-mini)


## 2.2 e 2.4 Ferramentas via function calling e o ciclo ReAct

Cada ferramenta é uma função Python com `@tool` e um `args_schema` Pydantic: o modelo recebe nome, descrição e schema (function calling) e decide quando chamar. As ferramentas da Câmara e da Brasil API fazem chamadas HTTP **reais** com `httpx`; as respostas são gravadas em `fixtures/` para o modo offline.

In [2]:
from src import tools

for t in tools.toolkit_basico():
    print(f"• {t.name}({', '.join(t.args)})\n    {t.description.strip().splitlines()[0]}")

• buscar_deputado(nome, uf)
    Busca deputados federais em exercicio pelo nome na API da Camara.
• listar_despesas(id_deputado, ano, mes, categoria)
    Lista as despesas da cota parlamentar (CEAP) de um deputado na API da Camara,
• dados_fornecedor(cnpj)
    Consulta a Brasil API e retorna razao social, situacao cadastral, CNAE
• calcular_estatisticas(valores, operacao)
    Calcula soma, media, mediana, maximo, minimo ou variacao percentual de uma


In [3]:
print(tools.buscar_deputado.invoke({"nome": "Tabata Amaral", "uf": "SP"}))
r = json.loads(tools.listar_despesas.invoke({"id_deputado": 204534, "ano": 2025, "categoria": "combustível"}))
print({k: v for k, v in r.items() if k not in ("amostra_despesas", "total_por_tipo")})
print(tools.calcular_estatisticas.invoke({"valores": [48808.83, 48849.91], "operacao": "variacao_pct"}))

id=204534 | Tabata Amaral | PSB-SP
{'id_deputado': 204534, 'ano': 2025, 'mes': None, 'total_registros': 201, 'total_valor': 48808.83}
variacao_pct = 0.08


### Agente ReAct com `create_agent` + `MemorySaver`

> **Nota de equivalência (rubrica 2.2).** O enunciado cita `initialize_agent` com `ZERO_SHOT_REACT_DESCRIPTION`. Essa função foi descontinuada no LangChain 1.x. O mesmo ciclo **raciocinar → agir → observar** é implementado por `create_agent` (sobre LangGraph): o modelo emite `tool_calls`, o grafo executa a ferramenta, devolve uma `ToolMessage` e o modelo decide o próximo passo até responder. O `MemorySaver` com `thread_id` é a memória da sessão. Abaixo o ciclo aparece nos logs do callback e na trajetória extraída do estado.

In [4]:
from src.agent import montar_agente, trajetoria
from src.chains import LogCallbackHandler

agente = montar_agente(
    com_sql=False, com_rag=False, com_memoria_episodica=False
)  # fase 2: só API + hardcoded
log = LogCallbackHandler()
cfg = {"configurable": {"thread_id": "fase2-demo"}, "callbacks": [log]}
estado = agente.invoke(
    {"messages": [("user", "Quanto a deputada Tabata Amaral gastou com combustível em 2025?")]}, config=cfg
)

print("RESPOSTA:", estado["messages"][-1].content, "\n")
print("TRAJETÓRIA (raciocínio -> ação -> observação):")
for passo in trajetoria(estado):
    if passo["tipo"] == "decisao":
        print(f"  AÇÃO      {passo['ferramenta']}({passo['args']})")
    else:
        print(f"  OBSERVAÇÃO {passo['ferramenta']}: {passo['conteudo'][:100]}...")

RESPOSTA: A deputada Tabata Amaral gastou um total de R$ 48.808,83 com combustível em 2025. 

Fonte: API da Câmara, despesas de 2025. 

TRAJETÓRIA (raciocínio -> ação -> observação):
  AÇÃO      buscar_deputado({'nome': 'Tabata Amaral'})
  OBSERVAÇÃO buscar_deputado: id=204534 | Tabata Amaral | PSB-SP...
  AÇÃO      listar_despesas({'id_deputado': 204534, 'ano': 2025, 'categoria': 'combustível'})
  OBSERVAÇÃO listar_despesas: {"id_deputado": 204534, "ano": 2025, "mes": null, "total_registros": 201, "total_valor": 48808.83, "...


In [5]:
# memória de curto prazo: o segundo turno herda o deputado e o ano
estado = agente.invoke({"messages": [("user", "E com passagens aéreas?")]}, config=cfg)
print(estado["messages"][-1].content)
print([(p["ferramenta"], p["args"]) for p in trajetoria(estado) if p["tipo"] == "decisao"])

A deputada Tabata Amaral gastou um total de R$ 46.072,94 com passagens aéreas em 2025.

Fonte: API da Câmara, despesas de 2025.
[('listar_despesas', {'id_deputado': 204534, 'ano': 2025, 'categoria': 'passagem aérea'})]


## 2.3 Chains sequenciais, paralelas e roteamento dinâmico

> **Nota de equivalência (rubrica 2.3).** `RouterChain`, `SequentialChain` e `LLMChain` são legados. O roteamento dinâmico é feito com uma **chain classificadora** (`temperature=0`, devolve `api | sql | regra | conversa`) e um `RunnableLambda` que despacha para a chain certa; a chain sequencial é um pipe LCEL e a paralela usa `RunnableParallel`.

In [6]:
from src.chains import chain_com_roteamento, chain_paralela, chain_roteador, chain_sequencial

print("--- roteador ---")
for p in [
    "Quanto a Tabata Amaral gastou em 2025?",
    "Qual a média do PSB com combustível?",
    "Qual o limite da cota em SP?",
    "Bom dia!",
]:
    print(f"{chain_roteador().invoke({'pergunta': p}):9} <- {p}")

print("\n--- sequencial: pergunta -> pedido -> API -> texto ---")
print(chain_sequencial().invoke("Quanto a Tabata Amaral gastou com combustível em 2025?"))

print("\n--- paralela: API da Câmara e SQL ao mesmo tempo ---")
print(
    chain_paralela().invoke(
        "Compare o gasto da Tabata Amaral do PSB com combustível em 2025 com a média do partido"
    )
)

print("\n--- roteada ---")
print(chain_com_roteamento().invoke("Qual o limite mensal da cota para SP?")[:300])

--- roteador ---


api       <- Quanto a Tabata Amaral gastou em 2025?


sql       <- Qual a média do PSB com combustível?


regra     <- Qual o limite da cota em SP?


conversa  <- Bom dia!

--- sequencial: pergunta -> pedido -> API -> texto ---


id=204534 | Tabata Amaral | PSB-SP: R$ 48.808,83 em 2025 (201 lancamentos). Fonte: API da Camara.

--- paralela: API da Câmara e SQL ao mesmo tempo ---


[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media FROM (SELECT id_deputado, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY id_deputado) t LIMIT 200
{'despesas': 'id=204534 | Tabata Amaral | PSB-SP: R$ 48.808,83 em 2025 (201 lancamentos). Fonte: API da Camara.', 'media_partido': "1 registros: [{'media': 48849.91}]"}

--- roteada ---


Limite mensal da cota para deputado de RR: R$ 51.406,33

Limite mensal da cota para deputado de RS: R$ 46.669,70

Limite mensal da cota para deputado de SC: R$ 45.671,58

Limite mensal da cota para deputado de SE: R$ 45.933,06

Limite mensal da cota para deputado de SP: R$ 42.837,33

Limite mensal d


## 2.5 Prompt PTCF, few-shot e chain-of-thought

O system prompt tem quatro blocos explícitos (Persona, Task, Context, Format). O few-shot usa `FewShotChatMessagePromptTemplate` com três exemplos pergunta → plano de ferramentas, incluindo o caso ambíguo dos homônimos. O chain-of-thought está na instrução "antes de escolher a ferramenta, escreva em uma linha o que precisa descobrir e em que ordem".

In [7]:
from src import prompts

print(prompts.SYSTEM_PROMPT_PTCF)

# PERSONA
Voce e o ApuraCota, analista de dados de uma redacao de jornalismo investigativo.
E rigoroso com numeros, cita sempre o periodo e a fonte, e prefere dizer "nao encontrei"
a inventar um valor.

# TASK
Responder perguntas de reporteres sobre a Cota para o Exercicio da Atividade Parlamentar
(CEAP) da Camara dos Deputados, decidindo a cada passo qual ferramenta usar:
1. 'buscar_deputado' para resolver nomes em id, partido e UF. SEMPRE chame a ferramenta antes de
   responder ou de pedir esclarecimento: nunca presuma de memoria que um nome e ambiguo ou qual e o id.
   Se a busca devolver mais de um deputado, liste as opcoes e pergunte qual e o certo.
2. 'listar_despesas' para despesas de UM deputado por ano/mes/categoria (API da Camara).
3. 'consultar_ceap' (SQL) para agregacoes: medias por partido ou UF, rankings de fornecedores,
   series mensais, comparacoes entre varios deputados.
4. 'calcular_estatisticas' para QUALQUER conta. Nunca some, divida ou compare de cabeca.
5. 'cons

In [8]:
print("--- few-shot (pergunta -> plano) ---")
for m in prompts.FEW_SHOT_PLANOS.format_messages():
    print(f"[{m.type}] {m.content}")

--- few-shot (pergunta -> plano) ---
[human] Quanto o deputado Fulano gastou com combustivel em 2025?
[ai] Plano: 1) buscar_deputado('Fulano') 2) listar_despesas(id, 2025, categoria='COMBUSTÍVEIS') 3) calcular_estatisticas(valores, 'soma') 4) responder com total, periodo e fonte.
[human] Compare esse gasto com a media do partido dele.
[ai] Plano: 1) usar partido ja conhecido no contexto 2) consultar_ceap(SELECT AVG por deputado do partido na categoria e ano) 3) calcular_estatisticas([media, gasto], 'variacao_pct') 4) responder.
[human] Gastos do Silva em 2025.
[ai] Plano: 1) buscar_deputado('Silva') retorna varios homonimos 2) NAO consultar despesas; perguntar ao usuario qual Silva (nome completo ou UF).


In [9]:
# o caso ambíguo do few-shot na prática: o agente NÃO consulta despesas sem resolver o homônimo
estado = agente.invoke(
    {"messages": [("user", "Gastos do Silva em 2025")]}, config={"configurable": {"thread_id": "homonimo"}}
)
print([p["ferramenta"] for p in trajetoria(estado) if p["tipo"] == "decisao"])
print(estado["messages"][-1].content)

['buscar_deputado']
Encontrei vários deputados com o nome "Silva":

1. Benedita da Silva - PT-RJ (id: 73701)
2. Cabo Gilberto Silva - PL-PB (id: 220574)
3. Marina Silva - REDE-SP (id: 220637)
4. Orlando Silva - PCdoB-SP (id: 178987)
5. Ribamar Silva - PODE-SP (id: 175765)
6. Zé Silva - UNIÃO-MG (id: 160632)

Qual deles você gostaria de consultar sobre os gastos em 2025?


## 2.6 Componentes de confiabilidade

| Componente | Implementação | Por quê |
|---|---|---|
| Parser com tratamento de erro | `PydanticOutputParser` + chain de reparo que captura `OutputParserException` e pede ao modelo para corrigir o JSON | saída estruturada quebrada não pode derrubar a apuração |
| Memória configurada | `trim_messages` (janela por tokens, preserva o system prompt, começa em mensagem humana) | listas de despesas são longas; sem poda o contexto estoura e o custo sobe |
| FAISS | índice do Ato da Mesa (chunks de 500/100), salvo com `save_local` e reaberto na fase 4 | busca semântica das regras com proveniência (chunk_id, score) |
| Callbacks | `LogCallbackHandler(BaseCallbackHandler)`: `on_chat_model_start`, `on_llm_end`, `on_tool_start/end/error` com duração | evidência do ciclo ReAct e material para o relatório e para diagnosticar latência |

In [10]:
from src.chains import parse_com_reparo

print(
    "válido  ->",
    parse_com_reparo('{"intencao": "consultar_gasto", "deputado": "Tabata Amaral", "ano": 2025}'),
)
print("quebrado->", parse_com_reparo('{"intencao": "voar", "ano": "dois mil"}'))

válido  -> intencao='consultar_gasto' deputado='Tabata Amaral' partido=None uf=None ano=2025 mes=None categoria=None dado_faltante=None
[parser] saida invalida, pedindo reparo (1): Failed to parse PedidoApuracao from completion {"intencao": "voar", "ano": "dois


quebrado-> intencao='consultar_gasto' deputado=None partido=None uf=None ano=None mes=None categoria=None dado_faltante=None


In [11]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

from src.memory import podar_historico

historico = [SystemMessage("system prompt")] + [
    m
    for i in range(6)
    for m in (HumanMessage(f"pergunta {i} " + "x" * 400), AIMessage(f"resposta {i} " + "y" * 400))
]
podado = podar_historico(historico, llm.chat_model(), max_tokens=500)
print(
    f"{len(historico)} mensagens -> {len(podado)} após trim_messages; primeira: {podado[0].type}, segunda: {podado[1].type}"
)

13 mensagens -> 5 após trim_messages; primeira: system, segunda: human


In [12]:
from src.data import carregar_indice, construir_indice

n = construir_indice()  # grava indices/ato_mesa (será o "índice existente" da fase 4)
indice = carregar_indice()
print(n, "chunks indexados")
for doc, score in indice.similarity_search_with_score("limite mensal da cota por unidade da federação", k=2):
    print(f"[{doc.metadata['chunk_id']}] L2={score:.3f} :: {doc.page_content[:110]!r}")

105 chunks indexados


[CHK_104] L2=0.767 :: 'Limite mensal da cota para deputado de RR: R$ 51.406,33\n\nLimite mensal da cota para deputado de RS: R$ 46.669,'
[CHK_101] L2=0.774 :: 'Limite mensal da cota para deputado de AC: R$ 50.426,26\n\nLimite mensal da cota para deputado de AL: R$ 46.737,'


In [13]:
print(log.como_json()[:1200], "...")

[
 {
  "hora": "19:43:06.379",
  "tipo": "llm_start",
  "n_mensagens": 2
 },
 {
  "hora": "19:43:07.562",
  "tipo": "llm_end",
  "duracao_s": 1.183,
  "tool_calls": [
   "buscar_deputado"
  ]
 },
 {
  "hora": "19:43:07.564",
  "tipo": "tool_start",
  "ferramenta": "buscar_deputado",
  "args": {
   "nome": "Tabata Amaral"
  }
 },
 {
  "hora": "19:43:07.566",
  "tipo": "tool_end",
  "duracao_s": 0.001,
  "saida": "id=204534 | Tabata Amaral | PSB-SP"
 },
 {
  "hora": "19:43:07.568",
  "tipo": "llm_start",
  "n_mensagens": 4
 },
 {
  "hora": "19:43:08.895",
  "tipo": "llm_end",
  "duracao_s": 1.327,
  "tool_calls": [
   "listar_despesas"
  ]
 },
 {
  "hora": "19:43:08.897",
  "tipo": "tool_start",
  "ferramenta": "listar_despesas",
  "args": {
   "id_deputado": 204534,
   "ano": 2025,
   "categoria": "combustível"
  }
 },
 {
  "hora": "19:43:08.907",
  "tipo": "tool_end",
  "duracao_s": 0.01,
  "saida": "{\"id_deputado\": 204534, \"ano\": 2025, \"mes\": null, \"total_registros\": 201, \"to

**Resultado da fase 2:** agente ReAct que resolve nomes na API da Câmara, lista despesas, faz contas por ferramenta determinística e mantém o contexto entre turnos; roteamento e chains compostas em LCEL; prompt PTCF com few-shot; parser tolerante a erro, janela de contexto podada, índice FAISS salvo e logs de callback.